# SENTIMENT + SUGGESTIONS WORKFLOW

In [30]:
# --- Libraries ---
import os
from langgraph.graph import StateGraph, START, END
from langchain_groq.chat_models import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict, Literal
from dotenv import load_dotenv

In [4]:
# load api key
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [39]:
# --- Define LLM ---
LLM = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0.5,
)

In [33]:
# --- Define State --- 
class State(TypedDict):
    text: str
    sentiment: Literal["Positive", "Negative"]
    suggestion: str

In [34]:
# --- Define functions ---
def classifiy_sentiment(state: State):
    messages = [
        SystemMessage(content = "You are an expert in classifying sentiments"),
        HumanMessage(content = f"Classify the sentiment: {state['text']} and answer in {state['sentiment']}")
    ]
    response = LLM.invoke(messages).content
    return {"sentiment": response}

In [35]:
# --- Define function ---
def check_sentiment(state: State) -> Literal["sentiment_suggestion"]:
    if state['sentiment'] == "Postive":
        return "positive"
    else:
        return "negative"

In [36]:
# --- Define function ---
def sentiment_suggestion(state: State):
    messages = [
        SystemMessage(content = "You are an amazing motivational speaker"),
        HumanMessage(content = f"Generate a short speech or line for the {state['sentiment']}")
    ]
    response = LLM.invoke(messages).content
    return {"suggestion": response}

In [37]:
# --- Build graph ---
graph = StateGraph(State)
graph.add_node("sentiment", classifiy_sentiment)
graph.add_node("suggestion", sentiment_suggestion)
graph.add_node("check_sentiment", check_sentiment)

graph.add_edge(START, "sentiment")
graph.add_conditional_edges("sentiment", check_sentiment, {"positive": "suggestion", "negative": "suggestion"})

graph.add_edge("suggestion", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer)

In [40]:
# --- Execute ---
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
initial_state = {
    "text": "This movie is amazing to watch",
    "sentiment": "",
    "suggestion": ""
}
print(workflow.invoke(initial_state, config))

{'text': 'This movie is amazing to watch', 'sentiment': 'The sentiment of the statement "This movie is amazing to watch" is: **POSITIVE**', 'suggestion': '"Ladies and gentlemen, I want to leave you with a thought tonight: just like a blockbuster movie that leaves us on the edge of our seats, life is a cinematic masterpiece waiting to unfold. And I truly believe, this movie we call life, is absolutely amazing to watch, with every twist and turn leading us to a greater purpose. So, let\'s grab the popcorn, sit back, and enjoy the incredible journey that is our lives!"'}


In [41]:
# --- Fetch latest state ---
workflow.get_state(config)

StateSnapshot(values={'text': 'This movie is amazing to watch', 'sentiment': 'The sentiment of the statement "This movie is amazing to watch" is: **POSITIVE**', 'suggestion': '"Ladies and gentlemen, I want to leave you with a thought tonight: just like a blockbuster movie that leaves us on the edge of our seats, life is a cinematic masterpiece waiting to unfold. And I truly believe, this movie we call life, is absolutely amazing to watch, with every twist and turn leading us to a greater purpose. So, let\'s grab the popcorn, sit back, and enjoy the incredible journey that is our lives!"'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0aa000-a8cd-6d33-8004-a43bcdeee482'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-10-15T19:49:20.221076+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0aa000-a132-6ea0-8003-1102564d4b83'}}, tasks=(), interrupts=())

In [44]:
# --- State history
config = {"configurable": {"thread_id": "1"}}
list(workflow.get_state_history(config))

[StateSnapshot(values={'text': 'This movie is amazing to watch', 'sentiment': 'The sentiment of the statement "This movie is amazing to watch" is: **POSITIVE**', 'suggestion': '"Ladies and gentlemen, I want to leave you with a thought tonight: just like a blockbuster movie that leaves us on the edge of our seats, life is a cinematic masterpiece waiting to unfold. And I truly believe, this movie we call life, is absolutely amazing to watch, with every twist and turn leading us to a greater purpose. So, let\'s grab the popcorn, sit back, and enjoy the incredible journey that is our lives!"'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0aa000-a8cd-6d33-8004-a43bcdeee482'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-10-15T19:49:20.221076+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0aa000-a132-6ea0-8003-1102564d4b83'}}, tasks=(), interrupts=()),
 StateSnapsh